In [50]:
import numpy as np
import pandas as pd
import os, re, csv, gc

import chime
import matplotlib.pyplot as plt

import torch
from torch import nn

chime.info()

In [43]:
keys = {'artist': 0, 'title': 1, 'link': 2, 'lyrics': 3}


def stream_samples():
    with open('data/songdata.csv', "r", newline='') as file:
        reader = csv.reader(file)
        next(reader)
        for row in reader:
            yield row


def get_sample(index: int):
    for i, item in enumerate(stream_samples()):
        if i == index:
            return item
    return None


def simplify_lyrics(lyrics: str):
    lyrics = re.sub(r" +", " ", lyrics)
    lyrics = re.sub(r"[^\w\n., ]", "", lyrics)
    return lyrics.lower().strip()


def lyrics_to_words(lyrics: str) -> list[str]:
    words = re.split(r" |(?=\n)|(?<=\n)", lyrics.strip())
    return [word for word in words if word != '']


def stream_words():
    for sample in stream_samples():
        if sample[keys['artist']] != 'Justin Bieber': continue
        lyrics = simplify_lyrics(sample[keys['lyrics']])
        yield lyrics_to_words(lyrics)

In [38]:
l = simplify_lyrics([sample[keys['lyrics']] for sample in stream_samples() if sample[keys['artist']] == 'Justin Bieber'][0])
print(l)

on the 4th day of christmas 
justin bieber gave to me 
4 concert tickets 
3 pairs of nikeys 
2 hair flips 
1 less lonely girl 
 
on the 6th day of christmas 
justin bieber gave to me 
6 cool scarfs 
5 pairs of supras 
4 concert tickets 
3 pairs of nikeys 
2 hair flips 
1 less lonely girl 
 
on the 9th day of christmas 
justin bieber gave to me 
9 no 1 songs 
8 plaid shirts 
7 new hats 
6 cool scarfs 
5 pairs of supras 
4 concert tickets 
3 pairs of nikeys 
2 hair flips 
1 less lonely girl


In [44]:
class Vocabulary:
    def __init__(self, min_freq=2):
        self.word2idx = {"<PAD>": 0, "<UNK>": 1}
        self.idx2word = {0: "<PAD>", 1: "<UNK>"}
        self.word_freq = {}
        self.min_freq = min_freq

    def build_vocab(self, songs):
        print("build_vocab: Starting")
        for song in songs:
            for word in song:
                self.word_freq[word] = self.word_freq.get(word, 0) + 1

        for word, freq in self.word_freq.items():
            if freq >= self.min_freq:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx] = word
        print("build_vocab: Ending")

    def encode(self, song):
        return [self.word2idx.get(w, 1) for w in song]

    def __len__(self):
        return len(self.word2idx)

In [45]:
vocabulary = Vocabulary()

In [46]:
vocabulary.build_vocab(stream_words())

build_vocab: Starting
build_vocab: Ending


In [51]:
class SongDataset(torch.utils.data.IterableDataset):
    def __init__(self, vocabulary, data_generator, seq_len=20):
        self.vocabulary = vocabulary
        self.data_generator = data_generator
        self.seq_len = seq_len

    def __iter__(self):
        for song in self.data_generator():
            song = self.vocabulary.encode(song)

            if len(song) < 2:  continue
            for i in range(len(song) - 1):
                start = max(0, i - self.seq_len)
                x = song[start:i + 1]
                y = song[i + 1]

                yield torch.tensor(x), torch.tensor(y)

            del song
            gc.collect()

In [52]:
def collate_fn(batch: tuple[torch.Tensor, torch.Tensor]):
    xs, ys = zip(*batch)
    max_len = max(len(x) for x in xs)
    padded = torch.zeros(len(xs), max_len, dtype=torch.long)
    for i, x in enumerate(xs):
        padded[i, :len(x)] = x
    ys = torch.stack(ys)
    return padded, ys

In [53]:
class LyricsModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=200, hidden_dim=256):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=2,
            bidirectional=True,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_dim * 2, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        outputs, _ = self.lstm(embedded)
        last_hidden = outputs[:, -1, :]
        logits = self.fc(last_hidden)
        return logits

In [64]:
def train_model(model, data_dir, vocab_size, epochs=5, batch_size=32):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # device = "cpu"
    model.to(device)

    dataset = SongDataset(vocabulary, stream_words)
    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        collate_fn=collate_fn
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    model.train()
    for epoch in range(epochs):
        total_loss = 0

        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")


In [55]:
model = LyricsModel(len(vocabulary))

In [56]:
# cpu time: 10 m 42 s
# train_model(model, "data_chunks", len(vocabulary))

Epoch 1 Loss: 7906.9723
Epoch 2 Loss: 6128.0614
Epoch 3 Loss: 4788.1110
Epoch 4 Loss: 3721.9717
Epoch 5 Loss: 2887.2752


In [82]:
# gpu time: 3 m 59 s
train_model(model, "data_chunks", len(vocabulary))

Epoch 1 Loss: 1035.8515
Epoch 2 Loss: 805.5054
Epoch 3 Loss: 731.4369
Epoch 4 Loss: 685.6643
Epoch 5 Loss: 632.4197


In [66]:
def generate_samples(
    model,
    vocab,
    start_text="life",
    max_length=50,
    temperature=1.0,
    top_k=None,
    num_samples=3
):
    device = next(model.parameters()).device
    model.eval()

    results = []

    for _ in range(num_samples):

        words = start_text.split()
        input_ids = [vocab.word2idx.get(w, 1) for w in words]

        for _ in range(max_length):

            x = torch.tensor([input_ids], dtype=torch.long).to(device)

            with torch.no_grad():
                logits = model(x)

            logits = logits / temperature
            probs = torch.nn.functional.softmax(logits, dim=-1)

            # 🔥 Top-k sampling
            if top_k is not None:
                top_probs, top_indices = torch.topk(probs, top_k)
                probs = torch.zeros_like(probs).scatter_(
                    -1, top_indices, top_probs
                )
                probs = probs / probs.sum(dim=-1, keepdim=True)

            next_token = torch.multinomial(probs, 1).item()

            input_ids.append(next_token)

            if len(input_ids) > 1 and next_token == 0:  # PAD
                break

        generated_words = [
            vocab.idx2word.get(idx, "<UNK>")
            for idx in input_ids
        ]

        results.append(" ".join(generated_words))

    return results


In [112]:
samples = generate_samples(
    model,
    vocabulary,
    start_text="fame feels cold",
    max_length=40,
    temperature=0.9,
    top_k=30,
    num_samples=5
)

for i, s in enumerate(samples):
    print(f"\nSample {i+1}:\n{s}")


Sample 1:
fame feels cold no matter what oh baby 
 
 if you let me inside your world 
 therell be peace on earth 
 by sweet <UNK> but my big <UNK> i got rid of you 
 i said girl, we can go

Sample 2:
fame feels cold no matter what they say 
 so just <UNK> me in the <UNK> yeah 
 no <UNK> <UNK> is <UNK> told her dont get nervous 
 i dont <UNK> <UNK> this <UNK> that thing in my <UNK> 
 i know

Sample 3:
fame feels cold no no matter what oh yeah 
 
 verse 2 
 everybody has their <UNK> 
 say a <UNK> and 
 as <UNK> what meant the time 
 like <UNK> and show this game for <UNK> 
 will show the

Sample 4:
fame feels cold no matter what 
 take it down to the top now 
 and the boys all that i cant find cause, 
 im the one you never leavin but my time of this thing is 
 can we go <UNK>

Sample 5:
fame feels cold no need to go 
 so can we can take it now 
 i just cant forget the world, know 
 show me what you say 
 lets go nowhere but up 
 
 but in the look <UNK> me


```
this dream is too good to be true girl
all you gotta do is read the signs
i just cant stop closer
this can make the finer things yeah
just let me work you listen
girl you know that i still
---
this dream is alive the im saying
all the girls are laughing, feelings
you know that we are
that we can go nowhere but up
---
its true that my life is true
we can make the sun shine in the moonlight
we can make those gray clouds turn to blue skies
i know its hard baby believe, me
that we can go nowhere but up
---
i found heaven in your arms
baby it sings to me and girl i cant find me young
im a money to be <UNK>

so why did you leave me so speechless
i cant believe you let it get to your
---
fame feels cold no matter what oh baby

if you let me inside your world
therell be peace on earth
by sweet <UNK> but my big <UNK> i got rid of you
i said girl, we can go
---
fame feels cold no need to go
so can we can take it now
i just cant forget the world, know
show me what you say
lets go nowhere but up
```